<a href="https://colab.research.google.com/github/r1shi-j/coa122-coursework/blob/main/122_coursework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subscriptionManager as smSL
import feedbackManager as fm
from datetime import datetime

# Check Feedback Manager Works.
# feedback_list = fm.load_feedback()
# print(feedback_list)  # Should print the list of feedback records.
# fm.add_feedback("dft01", 4, "Not bad", "Game_Feedback.txt")

# # Check Subscription Manager Works.
subscriptions = smSL.load_subscriptions()
print(smSL.check_subscription("lbro", subscriptions))
# # Should return True or False based on the current date.
print(smSL.get_rental_limit("Basic"))  # Should return 2.
print(smSL.get_rental_limit("Premium"))  # Should return 7.

True
2
7


In [ ]:
# fm.add_feedback("hi", 999, "Excel", "Game_Feedback.txt")
# feedback_list = fm.load_feedback(file_name="Game_Feedback.txt")
# print(feedback_list)

### System Skeleton
Show main menu
- check availability of tools
  - search for tools
- rent tools to customers
  - must have valid subscription
  - must have not reached their renting limit
  - tool must be available (and a valid tool)
  - return tools
  - search for tools
- maintain a database of all tools
  - search for tools
- pruning
  - reads Tool_Feedback to understand which tools aren't popular
-----------------------------------------------------
_Files to use_

`Tool_Info.txt`
- stores ToolID,Name,Size,PurchasePrice,PurchaseDate
- all tools
- read-only
  - list all tools
  - search tools given a search term
  - check if tool exists

`Tool_Feedback.txt`
- stores ToolID,Rating,Comments
- logs customer feedback about the tool
- read-append
  - log tool feedback
  - find the least popular tools

`Subscription_Info.txt`
- stores CustomerID,SubscriptionType,StartDate,EndDate
- all customers with a subscription
- read-only
  - check if customer has a subscription
  - check if customer has reached their renting limit

`Rental_History.txt`
- stores ToolID,CustomerID,RentalDate,ReturnDate
- logs all rentals
- read-append
  - check if tool is available
  - log a tool rental
  - log a tool return

`Booking.txt`
- stores DIYClass,BookingDate,Time
- all DIY class bookings?
- read-only?

-----------------------------------------------------

Make dummy data
- generate_id function for 4-char UUID

In [ ]:
# HELPER: load tools
def loadTools():
    tools = []
    with open("Tool_Info.txt", "r") as file:
        next(file)
        for line in file:
            line = line.strip()
            if line:
                id, name, size, pprice, pdate = line.split(',')
                tools.append({'id': id, 'name': name, 'size': size, 'purchasePrice': int(pprice), 'purchaseDate': datetime.strptime(pdate, '%Y-%m-%d')})
    return tools

In [ ]:
# list all tools
def listTools():
    tools = loadTools()
    for tool in tools:
        print(f"• {tool['name']}")

In [ ]:
# check if tool exists
def doesToolExist(toolID):
    tools = loadTools()
    for tool in tools:
        if tool['id'] == toolID:
            return True
    return False

In [ ]:
# search tools given a search term

In [ ]:
# check if a customer has a subscription
def doesCustomerHaveSubscription(customerID):
    subscriptions = smSL.load_subscriptions()
    return smSL.check_subscription(customerID, subscriptions)

In [63]:
# HELPER: load rental history
def loadRentalHistory():
    rentals = []
    with open("Rental_History.txt", "r") as file:
        next(file)
        for line in file:
            line = line.strip()
            if line:
                toolID, customerID, rentalDate, returnDate = line.split(',')
                rentals.append({'toolID': toolID, 'customerID': customerID, 'rentalDate': rentalDate, 'returnDate': returnDate})
    return rentals
# loadRentalHistory()

[{'toolID': '1',
  'customerID': 'lbro',
  'rentalDate': '2026-11-15',
  'returnDate': '2026-11-15'},
 {'toolID': '4',
  'customerID': 'lbro',
  'rentalDate': '2026-11-15',
  'returnDate': '2026-11-15'},
 {'toolID': '6',
  'customerID': 'lbro',
  'rentalDate': '2026-10-25',
  'returnDate': '2026-11-15'},
 {'toolID': '6',
  'customerID': 'lbro',
  'rentalDate': '2026-10-15',
  'returnDate': '2026-11-12'}]

In [ ]:
# check if a customer has reached their renting limit
def hasCustomerReachedRentingLimit(rentingCustomerID):
    count = 0
    rentals = loadRentalHistory()
    for rental in rentals:
        if rental['customerID'] == rentingCustomerID and datetime.strptime(rental['rentalDate'], '%Y-%m-%d') < datetime.now():
            count += 1
    subscriptions = smSL.load_subscriptions()
    subscriptionPlan = subscriptions[rentingCustomerID]['SubscriptionType']
    return count >= smSL.get_rental_limit(subscriptionPlan)

In [ ]:
# check if tool is available
### Question: is the return date always logged when a tool is rented, or will this field remain empty until the tool is returned
def isToolAvailable(toolID):
    rentals = loadRentalHistory()
    for rental in rentals:
        if rental['toolID'] == toolID and datetime.strptime(rental['returnDate'], '%Y-%m-%d') == None or datetime.strptime(rental['returnDate'], '%Y-%m-%d') > datetime.now():
            return False
    return True

In [ ]:
# log a tool rental
def logToolRental(toolID, customerID):
    with open("Rental_History.txt", "a") as file:
        file.write(f"{toolID},{customerID},{datetime.now().strftime('%Y-%m-%d')},\n")
# logToolRental("4", "lbro")

In [67]:
# log a tool returnal
# how to return a tool, customer id tool id but what if they ahve done it in the past, date? rent id?
def logToolReturn(toolID, customerID):
    rentals = loadRentalHistory()
    filtered = filter(lambda rental: rental['toolID'] == toolID and rental['customerID'] == customerID, rentals)
    sortedRentals = sorted(filtered, key=lambda rental: datetime.strptime(rental['rentalDate'], '%Y-%m-%d'), reverse=True)
    # print(sortedRentals)
    if sortedRentals:
        targetRental = sortedRentals[0]
        with open("Rental_History.txt", "r+") as file:
            header = next(file)
            newLines = []

            for line in file:
                line = line.strip()
                if line:
                    rowToolID, rowCustomerID, rowRentalDate, rowReturnDate = line.split(',')
                    currentRow = {'toolID': rowToolID, 'customerID': rowCustomerID, 'rentalDate': rowRentalDate, 'returnDate': rowReturnDate}
                    if currentRow == targetRental and not rowReturnDate:
                        # print("match")
                        currentRow['returnDate'] = datetime.now().strftime('%Y-%m-%d')
                    # else:
                        # print("no match")
                    newLines.append(currentRow)
            # print(newLines)
            file.seek(0)
            file.truncate()
            file.write(header)
            for row in newLines:
                file.write(f"{row['toolID']},{row['customerID']},{row['rentalDate']},{row['returnDate']}\n")
# logToolReturn("6", "lbro")


[{'toolID': '6', 'customerID': 'lbro', 'rentalDate': '2026-10-25', 'returnDate': '2026-11-15'}, {'toolID': '6', 'customerID': 'lbro', 'rentalDate': '2026-10-15', 'returnDate': '2026-11-12'}]
no match
no match
no match
no match
[{'toolID': '1', 'customerID': 'lbro', 'rentalDate': '2026-11-15', 'returnDate': '2026-11-15'}, {'toolID': '4', 'customerID': 'lbro', 'rentalDate': '2026-11-15', 'returnDate': '2026-11-15'}, {'toolID': '6', 'customerID': 'lbro', 'rentalDate': '2026-10-25', 'returnDate': '2026-11-15'}, {'toolID': '6', 'customerID': 'lbro', 'rentalDate': '2026-10-15', 'returnDate': '2026-11-12'}]


Testing data for Rental_History.txt
```
ToolID,CustomerID,RentalDate,ReturnDate
1,lbro,2026-11-15,2026-11-15
4,lbro,2026-11-15,2026-11-15
6,lbro,2026-10-25,
6,lbro,2026-10-15,2026-11-12
```

In [ ]:
print(loadTools())

[{'id': '1', 'name': 'Toolbox', 'size': 'Small', 'purchasePrice': 20, 'purchaseDate': datetime.datetime(2026, 8, 1, 0, 0)}, {'id': '2', 'name': 'Bulldozer', 'size': 'Large', 'purchasePrice': 3000, 'purchaseDate': datetime.datetime(2026, 8, 2, 0, 0)}, {'id': '3', 'name': 'Stump Grinder', 'size': 'Medium', 'purchasePrice': 1000, 'purchaseDate': datetime.datetime(2026, 8, 3, 0, 0)}]


In [ ]:
print("Welcome to the Tool Rental and DIY booking management system")
print("Please select from the following options")
opt = input("a) Check the availibility of tools\nr) Rent tools out to a customer\ne) Return a tool\nd) View a database of tools\n")

if opt == 'a':
    # check availbility of tools
    print("option a")

elif opt == 'r':
    # add tool renting and returning

    # rent a tool out to a customer
    customerID = input("Which customer wishes to rent a tool? (Enter their id)\n")

    # check if they have a valid subscription
    if doesCustomerHaveSubscription(customerID):
        print("Subscription confirmed")

        # check if they have reached their renting limit
        if not hasCustomerReachedRentingLimit(customerID):
            print("They have not reached their renting limit")
            toolID = input("Enter the tool id they wish to rent out.")

            # check if tool exists
            if doesToolExist(toolID):
                print("Tool exists")

                # check if tool is available
                if isToolAvailable(toolID):
                    print("Tool is available")

                    # log a tool rental
                    logToolRental(toolID, customerID)

                else:
                    print("Tool is not available")
            else:
                print("Tool does not exist")
        else:
            print("They have reached their renting limit")
    else:
        print(f"{customerID} does not have an onging subscription, so they will not be able to rent out any tools.")

elif opt == 'e':
    print("Return a tool")
    customerID = input("Which customer wishes to rent a tool? (Enter their id)\n")
    toolID = input("Enter the tool id they wish to rent out.")
    logToolReturn(toolID, customerID)

elif opt == 'd':
    # listing tools
    print("Here are all of the tools we have: ")
    listTools()

else:
    print("Invalid option")

Welcome to the Tool Rental and DIY booking management system
Please select from the following options
a) Check the availibility of tools
r) Rent tools out to a customer
e) Return a tool
d) View a database of tools
d
Here are all of the tools we have: 
• Toolbox
• Bulldozer
• Stump Grinder


1.1 Overview
Your task is to develop a simple tool rental and DIY booking management system. The system should enable
a store manager to manage tool rentals based on customer subscriptions and offer DIY classes.
The
manager should be able to:
- check the availability of tools,
- rent them out to customers
- maintain a
database of all tools in the inventory.

For each tool, the following information should be stored: name, size,
purchase price, purchase date, and a unique ID number to distinguish different copies of the same tool.

In [ ]:
import secrets
import string

def generate_id(length=4):
    alphabet = string.ascii_letters + string.digits
    return ''.join(secrets.choice(alphabet) for _ in range(length))

### TODO
# Cut lines to 80 chars max, via colab divider line